# 实验五：基于视觉的目标识别（实时摄像头版）

适用：**Jetson Nano / JupyterLab / USB 摄像头（V4L2）**。

实验任务：①实时图像采集；②BGR→HSV与Mask；③红黄绿蓝方块分类和轮廓；④不同光照下HSV阈值标定、保存记录。

**安全退出：** 每个实时显示单元默认运行20秒；也可使用 Jupyter 菜单 **内核 → 中断内核** 提前停止。不要重复同时打开摄像头的两个单元。**不使用 `cv2.imshow()`、`cv2.waitKey()`**。


## 0. 环境与参数

In [1]:
import cv2, time, csv, json
import numpy as np
from pathlib import Path
from datetime import datetime
from IPython.display import display, Image

OUTPUT_DIR = Path('week5_results')
OUTPUT_DIR.mkdir(exist_ok=True)
CAMERA_INDEX = 0
FRAME_WIDTH, FRAME_HEIGHT = 640, 480
DURATION_SECONDS = 20  # 每次演示时长；设为 0 可持续运行，使用“中断内核”退出
DISPLAY_FPS = 8       # 浏览器页面显示的目标最大刷新率；实际帧率取决于硬件/网络
JPEG_QUALITY = 75
LIGHT_CONDITION = 'normal'  # normal / dim / bright / shadow / reflective
MIN_AREA = 350
print('OpenCV版本:', cv2.__version__, '结果目录:', OUTPUT_DIR.resolve())

OpenCV版本: 4.5.1 结果目录: /home/jetson/Notebook/5.目标识别/week5_results


## 1. 实时摄像头画面

每一轮读取一帧并更新**同一个 Notebook 显示区域**。画面默认持续20秒；延长请修改上面的 `DURATION_SECONDS`。

> 中断内核后，`finally` 负责释放摄像头。通常不需要重启整个 kernel。

In [2]:
def open_camera():
    cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_V4L2)
    if not cap.isOpened():
        raise RuntimeError('打不开摄像头：检查 /dev/video0 或是否被其它程序占用')
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)
    return cap


def show_live(process=None, duration=None, save_last=False):
    """process(frame BGR) -> BGR图；使用浏览器内嵌图像更新，而不是 OpenCV GUI。"""
    duration = DURATION_SECONDS if duration is None else duration
    cap = open_camera()
    handle = None
    start = time.monotonic()
    shown = 0
    last_frame = None
    try:
        while duration <= 0 or time.monotonic() - start < duration:
            ok, frame = cap.read()
            if not ok:
                print('摄像头读取失败，停止运行')
                break
            last_frame = frame.copy()
            output = process(frame) if process else frame
            ok, encoded = cv2.imencode('.jpg', output, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
            if not ok:
                continue
            img = Image(data=encoded.tobytes())
            if handle is None:
                handle = display(img, display_id=True)
            else:
                handle.update(img)
            shown += 1
            time.sleep(max(0, 1 / max(DISPLAY_FPS, 1)))
    except KeyboardInterrupt:
        print('已手动中断')
    finally:
        cap.release()
        print('摄像头已释放，显示帧数：', shown)
        if save_last and last_frame is not None:
            path = OUTPUT_DIR / f'camera_{LIGHT_CONDITION}_{datetime.now():%Y%m%d_%H%M%S}.png'
            cv2.imwrite(str(path), last_frame)
            print('原始图片已保存：', path)

show_live(save_last=True)

摄像头读取失败，停止运行
摄像头已释放，显示帧数： 0


[ WARN:0] global /tmp/pip-req-build-q3gzfcr4/opencv/modules/videoio/src/cap_v4l.cpp (1004) tryIoctl VIDEOIO(V4L2:/dev/video0): select() timeout.


## 2. HSV颜色转换与Mask

OpenCV的HSV范围：**H=0~179，S=0~255，V=0~255**。红色通常跨越H=0的边界，因此用两组阈值。

下列只是**初始阈值**，务必根据实际彩色方块和光照标定。

In [3]:
# 每种颜色可以有多个HSV区间 (low, high)
HSV_RANGES = {
    'red': [([0, 95, 70], [10, 255, 255]), ([170, 95, 70], [179, 255, 255])],
    'yellow': [([18, 85, 75], [38, 255, 255])],
    'green': [([40, 65, 55], [85, 255, 255])],
    'blue': [([90, 75, 50], [135, 255, 255])]
}

# 定义图形绘制的BGR颜色
DRAW_COLORS = {'red': (0,0,255), 'yellow': (0,255,255), 'green': (0,255,0), 'blue': (255,0,0)}


def create_masks(hsv):
    masks = {}
    for name, ranges in HSV_RANGES.items():
        mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
        for low, high in ranges:
            mask = cv2.bitwise_or(mask, cv2.inRange(hsv, np.array(low, np.uint8), np.array(high, np.uint8)))
        kernel = np.ones((3,3), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        masks[name] = mask
    return masks


def mask_panel(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    masks = create_masks(hsv)
    tiles = []
    for name in ('red','yellow','green','blue'):
        tile = cv2.cvtColor(masks[name], cv2.COLOR_GRAY2BGR)
        cv2.putText(tile, name, (12,30), cv2.FONT_HERSHEY_SIMPLEX, .9, DRAW_COLORS[name], 2)
        tiles.append(tile)
    top = np.hstack(tiles[:2])
    bottom = np.hstack(tiles[2:])
    original = np.hstack([frame, cv2.cvtColor(hsv[:,:,0], cv2.COLOR_GRAY2BGR)])
    return np.vstack([original, top, bottom])

# 输出三行两列：原图/H通道、红黄Mask、绿蓝Mask
show_live(mask_panel)

摄像头读取失败，停止运行
摄像头已释放，显示帧数： 0


[ WARN:0] global /tmp/pip-req-build-q3gzfcr4/opencv/modules/videoio/src/cap_v4l.cpp (1004) tryIoctl VIDEOIO(V4L2:/dev/video0): select() timeout.


## 3. 四色方块识别与轮廓标注

对每种颜色Mask查找外轮廓，使用**面积 + 近似四边形 + 外接矩形宽高比 + 填充率**过滤部分非方块目标。透视、遮挡或真实尺寸变化会影响结果，需根据实物调整。

In [4]:
def detect_blocks(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    masks = create_masks(hsv)
    result = frame.copy()
    detections = []
    for name, mask in masks.items():
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < MIN_AREA:
                continue
            perimeter = cv2.arcLength(cnt, True)
            if perimeter == 0:
                continue
            poly = cv2.approxPolyDP(cnt, 0.04 * perimeter, True)
            x,y,w,h = cv2.boundingRect(cnt)
            if not (len(poly)==4 and cv2.isContourConvex(poly)):
                continue
            if not (0.6 <= w / max(h,1) <= 1.6):
                continue
            if area / max(w*h,1) < 0.55:
                continue
            cx, cy = x+w//2, y+h//2
            cv2.drawContours(result, [poly], -1, DRAW_COLORS[name], 2)
            cv2.putText(result, f'{name} ({cx},{cy})', (x,max(18,y-8)), cv2.FONT_HERSHEY_SIMPLEX,
                        .55, DRAW_COLORS[name], 2)
            cv2.circle(result, (cx,cy), 4, DRAW_COLORS[name], -1)
            detections.append({'color': name,'x':cx,'y':cy,'area':round(area,1)})
    cv2.putText(result, f'Blocks: {len(detections)}', (10,28), cv2.FONT_HERSHEY_SIMPLEX, .8, (255,255,255), 2)
    return result, detections

show_live(lambda frame: detect_blocks(frame)[0])

摄像头读取失败，停止运行
摄像头已释放，显示帧数： 0


[ WARN:0] global /tmp/pip-req-build-q3gzfcr4/opencv/modules/videoio/src/cap_v4l.cpp (1004) tryIoctl VIDEOIO(V4L2:/dev/video0): select() timeout.


## 4. HSV标定：ROI采样 + 阈值调整 + 不同光照记录

### 4.1 从采集帧中选择方块ROI

建议每次只让一块方块处于镜头中间；对采集到的图像用 `(x,y,w,h)` 指定方块内部区域，**尽量不要包含背景、反光边缘或阴影**。采样结果用于给出HSV范围参考，不会自动覆盖阈值。

In [5]:
def capture_frame():
    cap = open_camera()
    try:
        for _ in range(4):
            ok, frame = cap.read()
        if not ok:
            raise RuntimeError('读取摄像头失败')
        return frame
    finally:
        cap.release()

frame_sample = capture_frame()
display(Image(data=cv2.imencode('.jpg', frame_sample)[1].tobytes()))
print('图片宽高：', frame_sample.shape[1], frame_sample.shape[0])

[ WARN:0] global /tmp/pip-req-build-q3gzfcr4/opencv/modules/videoio/src/cap_v4l.cpp (1004) tryIoctl VIDEOIO(V4L2:/dev/video0): select() timeout.
[ WARN:0] global /tmp/pip-req-build-q3gzfcr4/opencv/modules/videoio/src/cap_v4l.cpp (1004) tryIoctl VIDEOIO(V4L2:/dev/video0): select() timeout.
[ WARN:0] global /tmp/pip-req-build-q3gzfcr4/opencv/modules/videoio/src/cap_v4l.cpp (1004) tryIoctl VIDEOIO(V4L2:/dev/video0): select() timeout.
[ WARN:0] global /tmp/pip-req-build-q3gzfcr4/opencv/modules/videoio/src/cap_v4l.cpp (1004) tryIoctl VIDEOIO(V4L2:/dev/video0): select() timeout.


RuntimeError: 读取摄像头失败

In [ ]:
# 根据上方图片修改方块ROI：x,y,w,h，单位像素
ROI = (180, 120, 100, 100)
x, y, w, h = ROI
roi = frame_sample[y:y+h, x:x+w]
if roi.size == 0:
    raise ValueError('ROI超出图像范围，请修改ROI')
hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
for i, channel in enumerate('HSV'):
    values = hsv_roi[:,:,i]
    print(f'{channel}: 5%={np.percentile(values,5):.1f}, 50%={np.median(values):.1f}, 95%={np.percentile(values,95):.1f}')
print('红色跨越H=0时分段处理，不要直接用单个最大最小H区间。')
display(Image(data=cv2.imencode('.jpg', roi)[1].tobytes()))

### 4.2 调整阈值并实时验证

直接修改第2节的 `HSV_RANGES`，然后**重新执行下面的单元**实时查看Mask和识别结果。此法不使用易崩溃的OpenCV Trackbar。

In [ ]:
# 举例：调节绿色的阈值（按现场实际情况修改后再运行）
# HSV_RANGES['green'] = [([40, 70, 60], [85, 255, 255])]

def combined_panel(frame):
    annotated, detected = detect_blocks(frame)
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    masks = create_masks(hsv)
    union = np.zeros(frame.shape[:2], np.uint8)
    for mask in masks.values():
        union = cv2.bitwise_or(union, mask)
    mask_color = cv2.cvtColor(union, cv2.COLOR_GRAY2BGR)
    cv2.putText(mask_color, 'Combined Mask', (10,25), cv2.FONT_HERSHEY_SIMPLEX,.7,(0,255,255),2)
    return np.hstack([annotated, mask_color])

show_live(combined_panel)

### 4.3 保存HSV参数与光照下的检测记录

更改 `LIGHT_CONDITION` 以区分 normal / dim / bright / shadow / reflective。将指定颜色方块置于画面中，填写真实颜色 `TARGET_COLOR`，本单元将采集一帧，保存识别图片、Mask、阈值JSON、CSV记录。**是否正确**根据真实颜色和目标数量判断，不能用识别结果自动冒充真值。

In [ ]:
LIGHT_CONDITION = 'normal'   # 按现场更改
TARGET_COLOR = 'red'       # 真实放入镜头的方块颜色：red/yellow/green/blue
EXPECTED_COUNT = 1        # 预期这种颜色的方块数量
NOTES = ''                # 例如：逆光、阴影、反光、背景相近

frame = capture_frame()
marked, detections = detect_blocks(frame)
current = [d for d in detections if d['color'] == TARGET_COLOR]
correct = len(current) == EXPECTED_COUNT and len(detections) == EXPECTED_COUNT
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
stem = f'{stamp}_{LIGHT_CONDITION}_{TARGET_COLOR}'
cv2.imwrite(str(OUTPUT_DIR / (stem + '_original.png')), frame)
cv2.imwrite(str(OUTPUT_DIR / (stem + '_detected.png')), marked)
hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
masks = create_masks(hsv)
for name,mask in masks.items():
    cv2.imwrite(str(OUTPUT_DIR / (stem + f'_{name}_mask.png')), mask)
with open(OUTPUT_DIR/'hsv_parameters.json','w',encoding='utf8') as f:
    json.dump(HSV_RANGES, f, ensure_ascii=False, indent=2)
row = {'time':stamp, 'light':LIGHT_CONDITION, 'target':TARGET_COLOR,
       'expected_count':EXPECTED_COUNT, 'detected_count':len(detections),
       'matched_count':len(current), 'correct':correct,
       'hsv_ranges':json.dumps(HSV_RANGES[TARGET_COLOR],ensure_ascii=False),
       'notes':NOTES}
log = OUTPUT_DIR/'recognition_results.csv'
with open(log,'a',newline='',encoding='utf-8-sig') as f:
    writer=csv.DictWriter(f,fieldnames=list(row))
    if f.tell()==0:
        writer.writeheader()
    writer.writerow(row)
print(row)
print('文件已保存到：',OUTPUT_DIR.resolve())
display(Image(data=cv2.imencode('.jpg',marked)[1].tobytes()))

## 5. 实验分析建议

对正常、暗光、强光场景分别完成第4.3节测试，并记录：识别数量、漏检/误检、阈值变化。

- 暗光通常使V减小，低于阈值可能漏检；
- 反光可能降低局部S值并提高V值，使颜色分割产生缺口；
- 阴影可能形成局部低V区域；
- 红色跨越H=0/179边界时应保留两段阈值；
- 若光源色温变化，色相H也可能变化，不能认为H绝对不变。

记录必须基于实际观察和测试结果，不要把上述一般规律直接填写为已经测得的结果。

**本Notebook不包含实验第5项PyTorch手势识别和机械臂动作控制。**